# Deepfake detection: CLIP ViT-L/14 + LoRA

Frozen CLIP backbone with LoRA adapters, trained only on the hackathon data with grouped folds
(near-duplicate / real-fake pairs kept within one fold) and robustness augmentations.
Source files are embedded below via `%%writefile` (generated by `kaggle/make_notebook.py`).

## Config: edit and run

In [ ]:
FOLDS  = [0]                                      # e.g. [0,1,2,3,4] for full CV
ARCH   = "vit_large_patch14_clip_224.openai"      # or ...clip_224.dfn2b / vit_large_patch14_dinov2.lvd142m
COMMON = "--epochs 5 --bs 32 --grad_ckpt"
# experiment name -> extra train.py args (all compared on the same folds).
# Baseline from the previous run (final-layer CLS, 10 ep, BCE): fold-0 AUC 0.9134.
RUNS = {
    "ml5":      "--feat_layers auto",                       # multi-layer CLS features
    "ml5_drop": "--feat_layers auto --drop_rate 0.08",      # + drop 8% highest-loss samples/batch
    "ml5_gce":  "--feat_layers auto --loss gce",            # + Generalized Cross-Entropy
}
RUN_TEST_INFERENCE = True   # bundle the best run's folds + time inference on the test set

In [ ]:
!pip install -q timm==1.0.31
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Source files

In [ ]:
import os
os.makedirs('code/src', exist_ok=True)

In [ ]:
%%writefile code/src/__init__.py
"""Deepfake detector package: LoRA, model, data, degradations."""

In [ ]:
%%writefile code/src/lora.py
"""Minimal LoRA (Hu et al., 2021) for nn.Linear layers.

Written in-house (no peft dependency) so that adapters can be merged back into
plain nn.Linear weights after training. The merged model is an ordinary timm
model: inference needs no LoRA code and pays zero extra latency.
"""
from __future__ import annotations

import math
from typing import Iterable

import torch
import torch.nn as nn


class LoRALinear(nn.Module):
    """y = W x + b + (alpha / r) * B A dropout(x), with W, b frozen."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float = 0.0):
        super().__init__()
        self.base = base
        self.r = r
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))  # zero init -> starts as identity
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        for p in self.base.parameters():
            p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lora = (self.dropout(x) @ self.lora_A.t()) @ self.lora_B.t()
        return self.base(x) + lora * self.scaling

    @torch.no_grad()
    def merged(self) -> nn.Linear:
        out = nn.Linear(self.base.in_features, self.base.out_features,
                        bias=self.base.bias is not None)
        delta = (self.lora_B.float() @ self.lora_A.float()) * self.scaling
        out.weight.copy_(self.base.weight.float() + delta)
        if self.base.bias is not None:
            out.bias.copy_(self.base.bias.float())
        return out.to(self.base.weight.device)


def _set_submodule(root: nn.Module, name: str, new: nn.Module) -> None:
    parent_name, _, child = name.rpartition(".")
    parent = root.get_submodule(parent_name) if parent_name else root
    setattr(parent, child, new)


def inject_lora(model: nn.Module, targets: Iterable[str], r: int, alpha: float,
                dropout: float = 0.0) -> list[str]:
    """Wrap every nn.Linear whose qualified name ends with one of `targets`."""
    targets = tuple(targets)
    names = [n for n, m in model.named_modules()
             if isinstance(m, nn.Linear) and n.endswith(targets)]
    for n in names:
        _set_submodule(model, n, LoRALinear(model.get_submodule(n), r, alpha, dropout))
    return names


def merge_lora(model: nn.Module) -> int:
    """Replace every LoRALinear with an equivalent plain nn.Linear (in place)."""
    names = [n for n, m in model.named_modules() if isinstance(m, LoRALinear)]
    for n in names:
        _set_submodule(model, n, model.get_submodule(n).merged())
    return len(names)

In [ ]:
%%writefile code/src/degradations.py
"""Image degradations shared by training augmentation and the robustness harness.

The hidden test "variants" are undisclosed, so we train with (and validate
against) the common real-world corruptions: re-compression, blur, sensor noise,
resolution loss, colour shifts, grayscale and occlusion. The training images
already contain noise / cutout / colour-block corruptions in BOTH classes, so
these are label-independent nuisances the model must learn to ignore.
"""
from __future__ import annotations

import io
import random

import numpy as np
from PIL import Image, ImageEnhance, ImageFilter


def jpeg(img: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=int(quality))
    buf.seek(0)
    return Image.open(buf).convert("RGB")


def blur(img: Image.Image, sigma: float) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius=sigma))


def noise(img: Image.Image, std: float, rng: random.Random) -> Image.Image:
    arr = np.asarray(img, dtype=np.float32)
    g = np.random.default_rng(rng.randrange(2**31)).normal(0.0, std, arr.shape)
    return Image.fromarray(np.clip(arr + g, 0, 255).astype(np.uint8))


def downscale(img: Image.Image, factor: float) -> Image.Image:
    w, h = img.size
    small = img.resize((max(8, int(w * factor)), max(8, int(h * factor))), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR)


def color(img: Image.Image, b: float, c: float, s: float) -> Image.Image:
    img = ImageEnhance.Brightness(img).enhance(b)
    img = ImageEnhance.Contrast(img).enhance(c)
    return ImageEnhance.Color(img).enhance(s)


def grayscale(img: Image.Image) -> Image.Image:
    return img.convert("L").convert("RGB")


def cutout(img: Image.Image, frac: float, rng: random.Random) -> Image.Image:
    w, h = img.size
    cw, ch = int(w * frac), int(h * frac)
    x, y = rng.randrange(0, w - cw + 1), rng.randrange(0, h - ch + 1)
    arr = np.array(img)
    arr[y:y + ch, x:x + cw] = rng.randrange(0, 256)
    return Image.fromarray(arr)


# ── Training-time random corruption ───────────────────────────────────────────
def random_degrade(img: Image.Image, rng: random.Random, strength: float = 1.0) -> Image.Image:
    """Apply a random subset of corruptions in random order."""
    ops = []
    if rng.random() < 0.30 * strength:
        ops.append(lambda im: jpeg(im, rng.randint(30, 95)))
    if rng.random() < 0.20 * strength:
        ops.append(lambda im: blur(im, rng.uniform(0.2, 2.0)))
    if rng.random() < 0.20 * strength:
        ops.append(lambda im: noise(im, rng.uniform(2.0, 15.0), rng))
    if rng.random() < 0.20 * strength:
        ops.append(lambda im: downscale(im, rng.uniform(0.35, 0.9)))
    if rng.random() < 0.30 * strength:
        ops.append(lambda im: color(im, rng.uniform(0.7, 1.3), rng.uniform(0.7, 1.3),
                                    rng.uniform(0.6, 1.4)))
    if rng.random() < 0.05 * strength:
        ops.append(grayscale)
    if rng.random() < 0.15 * strength:
        ops.append(lambda im: cutout(im, rng.uniform(0.1, 0.3), rng))
    rng.shuffle(ops)
    for op in ops:
        img = op(img)
    return img


# ── Fixed variants for the robustness harness (proxy for hidden test variants) ─
VARIANTS = {
    "clean":      lambda im, rng: im,
    "jpeg70":     lambda im, rng: jpeg(im, 70),
    "jpeg40":     lambda im, rng: jpeg(im, 40),
    "blur1.5":    lambda im, rng: blur(im, 1.5),
    "noise10":    lambda im, rng: noise(im, 10.0, rng),
    "down0.5":    lambda im, rng: downscale(im, 0.5),
    "color":      lambda im, rng: color(im, 1.2, 0.8, 1.3),
    "gray":       lambda im, rng: grayscale(im),
    "cutout0.2":  lambda im, rng: cutout(im, 0.2, rng),
    "combo":      lambda im, rng: jpeg(downscale(blur(im, 0.8), 0.6), 60),
}

In [ ]:
%%writefile code/src/model.py
"""Detector = timm backbone (CLIP / DINOv2 / SigLIP / ConvNeXt ...) + linear head.

Two feature modes:
  * feat_layers=None  -> the backbone's own pooled output (final CLS token after the last norm).
  * feat_layers=[...] -> CLS tokens taken after several transformer blocks, each LayerNorm-ed, then
    concatenated. Intermediate CLIP layers keep low-level texture/blending cues that the last,
    semantics-oriented layer discards (cf. RINE, ECCV 2024), which matters for forgery detection.
"""
from __future__ import annotations

import timm
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

from .lora import inject_lora

# Suffixes of the nn.Linear layers that receive LoRA adapters (timm ViT naming).
LORA_TARGETS_ATTN = ("attn.qkv", "attn.proj")
LORA_TARGETS_ALL = LORA_TARGETS_ATTN + ("mlp.fc1", "mlp.fc2")


def parse_feat_layers(spec: str | None, depth: int) -> list[int] | None:
    """'' / None -> None; 'auto' -> every 4th block ending at the last; '7,15,23' -> explicit."""
    if not spec:
        return None
    if spec == "auto":
        return list(range(depth - 1, -1, -4))[::-1]
    layers = sorted({int(s) % depth for s in spec.split(",")})
    return layers


class Detector(nn.Module):
    def __init__(self, arch: str, pretrained: bool = True, drop: float = 0.1,
                 img_size: int | None = None, feat_layers: list[int] | str | None = None):
        super().__init__()
        kw = {"img_size": img_size} if img_size else {}  # e.g. run DINOv2 at 224 instead of 518
        self.backbone = timm.create_model(arch, pretrained=pretrained, num_classes=0, **kw)
        if isinstance(feat_layers, str):
            feat_layers = parse_feat_layers(feat_layers, len(self.backbone.blocks))
        self.feat_layers = list(feat_layers) if feat_layers else None
        self.grad_ckpt = False
        dim = self.backbone.num_features
        if self.feat_layers:
            self.layer_norms = nn.ModuleList(nn.LayerNorm(dim) for _ in self.feat_layers)
            dim *= len(self.feat_layers)
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(dim, 1))

    def features(self, x: torch.Tensor) -> torch.Tensor:
        if not self.feat_layers:
            return self.backbone(x)
        b = self.backbone  # mirrors timm VisionTransformer.forward_features
        x = b.norm_pre(b.patch_drop(b._pos_embed(b.patch_embed(x))))
        wanted, cls = set(self.feat_layers), []
        for i, blk in enumerate(b.blocks):
            if self.grad_ckpt and self.training:
                x = checkpoint(blk, x, use_reentrant=False)
            else:
                x = blk(x)
            if i in wanted:
                cls.append(x[:, 0])
        return torch.cat([ln(c) for ln, c in zip(self.layer_norms, cls)], dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x)).squeeze(1)  # logit of P(fake)

    def set_grad_checkpointing(self, enable: bool = True):
        self.grad_ckpt = enable
        self.backbone.set_grad_checkpointing(enable)


def build_lora_detector(arch: str, pretrained: bool, r: int, alpha: float, lora_dropout: float,
                        targets=LORA_TARGETS_ALL, train_norms: bool = True,
                        head_drop: float = 0.1, img_size: int | None = None,
                        feat_layers: str | None = None) -> tuple[Detector, dict]:
    """Freeze the backbone, add LoRA adapters, keep head (+ optionally LayerNorms) trainable."""
    model = Detector(arch, pretrained=pretrained, drop=head_drop, img_size=img_size,
                     feat_layers=feat_layers)
    for p in model.backbone.parameters():
        p.requires_grad = False
    wrapped = inject_lora(model.backbone, targets, r=r, alpha=alpha, dropout=lora_dropout)
    if not wrapped:
        raise ValueError(f"No LoRA target layers matched in {arch}; check `targets`.")

    groups = {"lora": [], "norm": [], "head": list(model.head.parameters())}
    if model.feat_layers:
        groups["head"] += list(model.layer_norms.parameters())
    for n, p in model.backbone.named_parameters():
        if "lora_" in n:
            p.requires_grad = True
            groups["lora"].append(p)
        elif train_norms and ".norm" in f".{n}" and p.ndim == 1:
            p.requires_grad = True
            groups["norm"].append(p)
    return model, groups


def data_config(model: Detector, img_size: int | None = None) -> dict:
    cfg = timm.data.resolve_data_config({}, model=model.backbone)
    return {"mean": list(cfg["mean"]), "std": list(cfg["std"]),
            "img_size": int(img_size or cfg["input_size"][-1])}

In [ ]:
%%writefile code/src/data.py
"""Datasets: augmented training set and deterministic (optionally degraded) eval set."""
from __future__ import annotations

import random

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms import functional as TF

from .degradations import VARIANTS, random_degrade


def to_tensor(img: Image.Image, img_size: int, mean, std) -> torch.Tensor:
    if img.size != (img_size, img_size):
        img = img.resize((img_size, img_size), Image.BICUBIC)
    return TF.normalize(TF.to_tensor(img), mean, std)


class TrainDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int, mean, std, aug_strength: float = 1.0):
        self.paths = df["path"].tolist()
        self.labels = df["label"].astype("float32").tolist()
        self.img_size, self.mean, self.std = img_size, mean, std
        self.aug_strength = aug_strength

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        # torch.initial_seed() differs per worker and per epoch -> fresh but reproducible aug.
        rng = random.Random((torch.initial_seed() + i * 7919) % (2**63))
        img = Image.open(self.paths[i]).convert("RGB")
        if rng.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if rng.random() < 0.5:  # mild random-resized crop (faces are already centred)
            w, h = img.size
            s = rng.uniform(0.85, 1.0)
            cw, ch = int(w * s), int(h * s)
            x, y = rng.randint(0, w - cw), rng.randint(0, h - ch)
            img = img.crop((x, y, x + cw, y + ch))
        img = random_degrade(img, rng, self.aug_strength)
        return to_tensor(img, self.img_size, self.mean, self.std), torch.tensor(self.labels[i])


class EvalDataset(Dataset):
    """Deterministic: the same image always receives the same degradation."""

    def __init__(self, df: pd.DataFrame, img_size: int, mean, std, variant: str = "clean",
                 seed: int = 0):
        self.paths = df["path"].tolist()
        self.labels = df["label"].astype("float32").tolist()
        self.img_size, self.mean, self.std = img_size, mean, std
        self.variant, self.seed = variant, seed

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        rng = random.Random(self.seed * 100003 + i)
        img = Image.open(self.paths[i]).convert("RGB")
        if self.variant == "mixed":  # one random named variant per image
            name = rng.choice([k for k in VARIANTS if k != "clean"])
        else:
            name = self.variant
        img = VARIANTS[name](img, rng)
        return to_tensor(img, self.img_size, self.mean, self.std), torch.tensor(self.labels[i])

In [ ]:
%%writefile code/train.py
"""LoRA fine-tuning of a frozen foundation backbone (default: CLIP ViT-L/14) for deepfake detection.

One run = one fold. Outputs in <out_dir>/<run_name>/fold<k>/:
    member.pt      merged (LoRA folded into weights) fp16 model, ready for the inference bundle
    oof.csv        out-of-fold P(fake) on the clean validation fold
    robustness.json  per-degradation validation AUC (proxy for the hidden test variants)
    history.csv    per-epoch loss / AUC (training curves for the report)

Example (Kaggle / Colab T4):
    python train.py --fold 0 --arch vit_large_patch14_clip_224.openai --bs 32 --epochs 10
"""
from __future__ import annotations

import argparse
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

from src.data import EvalDataset, TrainDataset
from src.degradations import VARIANTS
from src.lora import merge_lora
from src.model import LORA_TARGETS_ALL, LORA_TARGETS_ATTN, build_lora_detector, data_config


def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument("--folds_csv", default="splits/folds.csv")
    ap.add_argument("--fold", type=int, default=0)
    ap.add_argument("--data_dir", default=None,
                    help="dataset root containing train/real, train/fake (overrides paths in folds_csv)")
    ap.add_argument("--full", action="store_true", help="train on ALL data (no validation)")
    ap.add_argument("--arch", default="vit_large_patch14_clip_224.openai")
    ap.add_argument("--img_size", type=int, default=None)
    ap.add_argument("--no_pretrained", action="store_true", help="random init (smoke tests only)")
    ap.add_argument("--feat_layers", default="",
                    help="'' = final pooled token; 'auto' = CLS of every 4th block; or '7,15,23'")
    # LoRA
    ap.add_argument("--r", type=int, default=16)
    ap.add_argument("--alpha", type=float, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--targets", choices=["attn", "all"], default="all")
    ap.add_argument("--no_train_norms", action="store_true")
    # optimisation
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--bs", type=int, default=32)
    ap.add_argument("--accum", type=int, default=1)
    ap.add_argument("--lr", type=float, default=3e-4, help="LoRA + LayerNorm LR")
    ap.add_argument("--head_lr", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--warmup", type=float, default=0.05, help="fraction of steps")
    ap.add_argument("--label_smoothing", type=float, default=0.05, help="BCE only")
    # label-noise robustness
    ap.add_argument("--loss", choices=["bce", "gce"], default="bce",
                    help="gce = Generalized Cross-Entropy (Zhang & Sabuncu 2018), noise-tolerant")
    ap.add_argument("--gce_q", type=float, default=0.7)
    ap.add_argument("--drop_rate", type=float, default=0.0,
                    help="small-loss trick: drop this fraction of highest-loss samples per batch")
    ap.add_argument("--drop_warmup", type=float, default=1.0,
                    help="epochs over which drop_rate ramps up linearly from 0")
    ap.add_argument("--head_drop", type=float, default=0.1)
    ap.add_argument("--aug_strength", type=float, default=1.0)
    ap.add_argument("--grad_ckpt", action="store_true")
    # misc
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--limit", type=int, default=0, help="subsample N train/val images (smoke test)")
    ap.add_argument("--out_dir", default="outputs")
    ap.add_argument("--run_name", default="clip_l14_lora")
    return ap.parse_args()


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def predict(model, loader, device, amp) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            logit = model(x) + model(torch.flip(x, dims=[3]))  # hflip TTA (same as inference)
        probs.append(torch.sigmoid(logit.float() / 2).cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(probs), np.concatenate(labels)


def per_sample_loss(logits, y, args):
    if args.loss == "gce":
        p_true = torch.sigmoid(logits * (2 * y - 1)).clamp_min(1e-6)  # prob. of the given label
        return (1 - p_true.pow(args.gce_q)) / args.gce_q
    y_s = y * (1 - args.label_smoothing) + 0.5 * args.label_smoothing
    return F.binary_cross_entropy_with_logits(logits, y_s, reduction="none")


def reduce_loss(losses, rate):
    """Average the (1 - rate) smallest per-sample losses (likely-clean samples)."""
    if rate <= 0:
        return losses.mean()
    keep = max(1, int(round(losses.numel() * (1 - rate))))
    return torch.topk(losses, keep, largest=False).values.mean()


def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan")


def main():
    args = parse_args()
    seed_everything(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    amp = device == "cuda"
    out = Path(args.out_dir) / args.run_name / ("full" if args.full else f"fold{args.fold}")
    out.mkdir(parents=True, exist_ok=True)
    (out / "args.json").write_text(json.dumps(vars(args), indent=2))

    # ── data ──
    df = pd.read_csv(args.folds_csv)
    if args.data_dir:
        sub = df.label.map({0: "real", 1: "fake"})
        df["path"] = [str(Path(args.data_dir) / "train" / s / f"{i}.jpg") for s, i in zip(sub, df.image_id)]
    tr_df = df if args.full else df[df.fold != args.fold]
    va_df = df.iloc[:0] if args.full else df[df.fold == args.fold]
    if args.limit:
        tr_df = tr_df.sample(min(args.limit, len(tr_df)), random_state=args.seed)
        va_df = va_df.sample(min(args.limit, len(va_df)), random_state=args.seed)

    # ── model ──
    targets = LORA_TARGETS_ALL if args.targets == "all" else LORA_TARGETS_ATTN
    model, groups = build_lora_detector(
        args.arch, pretrained=not args.no_pretrained, r=args.r, alpha=args.alpha,
        lora_dropout=args.lora_dropout, targets=targets, train_norms=not args.no_train_norms,
        head_drop=args.head_drop, img_size=args.img_size, feat_layers=args.feat_layers)
    if args.grad_ckpt:
        model.set_grad_checkpointing(True)
    model.to(device)
    dcfg = data_config(model, args.img_size)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"device={device} arch={args.arch} img={dcfg['img_size']} "
          f"trainable={n_train/1e6:.2f}M / {n_total/1e6:.1f}M ({100*n_train/n_total:.2f}%) "
          f"train={len(tr_df)} val={len(va_df)} feat_layers={model.feat_layers} "
          f"loss={args.loss} drop_rate={args.drop_rate}")

    tr_dl = DataLoader(TrainDataset(tr_df, dcfg["img_size"], dcfg["mean"], dcfg["std"],
                                    args.aug_strength),
                       batch_size=args.bs, shuffle=True, num_workers=args.workers,
                       pin_memory=amp, drop_last=True, persistent_workers=args.workers > 0)

    def eval_loader(variant):
        ds = EvalDataset(va_df, dcfg["img_size"], dcfg["mean"], dcfg["std"], variant, args.seed)
        return DataLoader(ds, batch_size=args.bs * 2, shuffle=False, num_workers=args.workers,
                          pin_memory=amp)

    # ── optimiser / schedule ──
    opt = torch.optim.AdamW([
        {"params": groups["lora"], "lr": args.lr, "weight_decay": args.wd},
        {"params": groups["norm"], "lr": args.lr, "weight_decay": 0.0},
        {"params": groups["head"], "lr": args.head_lr, "weight_decay": args.wd},
    ])
    total = max(1, args.epochs * len(tr_dl) // args.accum)
    warm = max(1, int(args.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}

    best, best_row, history = -1.0, {}, []
    for epoch in range(args.epochs):
        model.train()
        t0, losses = time.time(), []
        opt.zero_grad(set_to_none=True)
        for step, (x, y) in enumerate(tr_dl):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            rate = args.drop_rate * min(1.0, (epoch + step / len(tr_dl)) / max(args.drop_warmup, 1e-8))
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                logits = model(x).float()
            loss = reduce_loss(per_sample_loss(logits, y, args), rate)
            scaler.scale(loss / args.accum).backward()
            if (step + 1) % args.accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
            losses.append(loss.item())

        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "lr": sched.get_last_lr()[0]}
        if len(va_df):
            p, y = predict(model, eval_loader("clean"), device, amp)
            pm, ym = predict(model, eval_loader("mixed"), device, amp)
            row.update(val_loss=float(F.binary_cross_entropy(torch.tensor(p).float().clamp(1e-6, 1 - 1e-6),
                                                             torch.tensor(y).float())),
                       val_auc=safe_auc(y, p), val_auc_degraded=safe_auc(ym, pm))
            row["score"] = 0.5 * (row["val_auc"] + row["val_auc_degraded"])
        else:
            row["score"] = epoch  # full-data run: keep the last epoch
        row["sec"] = round(time.time() - t0, 1)
        history.append(row)
        pd.DataFrame(history).to_csv(out / "history.csv", index=False)
        print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}))
        if row["score"] > best:
            best, best_row = row["score"], dict(row)
            torch.save({n: p.detach().cpu() for n, p in model.state_dict().items() if n in trainable},
                       out / "best_trainable.pt")

    # ── final: reload best adapters, robustness report, OOF preds ──
    model.load_state_dict(torch.load(out / "best_trainable.pt"), strict=False)
    if len(va_df):
        report = {}
        for v in VARIANTS:
            p, y = predict(model, eval_loader(v), device, amp)
            report[v] = safe_auc(y, p)
            if v == "clean":
                pd.DataFrame({"image_id": va_df.image_id.values, "label": y.astype(int),
                              "confidence": p, "fold": args.fold}).to_csv(out / "oof.csv", index=False)
        deg = [a for k, a in report.items() if k != "clean"]
        report["mean_degraded"], report["worst_degraded"] = float(np.mean(deg)), float(np.min(deg))
        (out / "robustness.json").write_text(json.dumps(report, indent=2))
        print("robustness:", json.dumps({k: round(v, 4) for k, v in report.items()}))

    n_merged = merge_lora(model.backbone)
    model.set_grad_checkpointing(False)
    torch.save({
        "arch": args.arch, "img_size": dcfg["img_size"], "mean": dcfg["mean"], "std": dcfg["std"],
        "head_drop": args.head_drop, "feat_layers": model.feat_layers,
        "state_dict": {k: v.detach().cpu().half() for k, v in model.state_dict().items()},
        "metrics": best_row,
    }, out / "member.pt")
    print(f"merged {n_merged} LoRA layers -> {out / 'member.pt'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile code/build_bundle.py
"""Pack trained members (folds and/or architectures) into ONE weights file for inference_script.py.

The decision threshold is chosen on out-of-fold predictions (Youden's J), never on test data.

    python build_bundle.py --members "outputs/clip_l14_lora/fold*/member.pt" \
                           --oof "outputs/clip_l14_lora/fold*/oof.csv" --out model_weights.pt
"""
from __future__ import annotations

import argparse
import glob

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, roc_auc_score, roc_curve


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--members", nargs="+", required=True, help="member.pt paths or globs")
    ap.add_argument("--weights", nargs="*", type=float, default=None, help="per-member weights")
    ap.add_argument("--oof", nargs="*", default=[], help="oof.csv paths or globs (for threshold)")
    ap.add_argument("--out", default="model_weights.pt")
    args = ap.parse_args()

    paths = sorted(p for g in args.members for p in glob.glob(g))
    assert paths, "no member files matched"
    w = args.weights or [1.0] * len(paths)
    assert len(w) == len(paths), "need one weight per member"

    members = []
    for p, wi in zip(paths, w):
        m = torch.load(p, map_location="cpu", weights_only=False)
        m["weight"] = wi
        members.append(m)
        print(f"+ {p}  arch={m['arch']}  weight={wi}  metrics={m.get('metrics', {})}")

    threshold, oof_report = 0.5, {}
    oof_paths = sorted(p for g in args.oof for p in glob.glob(g))
    if oof_paths:
        oof = pd.concat([pd.read_csv(p) for p in oof_paths])
        oof = oof.groupby("image_id").agg(label=("label", "first"), confidence=("confidence", "mean"))
        y, p = oof.label.values, oof.confidence.values
        fpr, tpr, thr = roc_curve(y, p)
        threshold = float(np.clip(thr[np.argmax(tpr - fpr)], 0.0, 1.0))
        oof_report = {"oof_auc": float(roc_auc_score(y, p)),
                      "oof_f1": float(f1_score(y, p >= threshold)),
                      "oof_acc": float(((p >= threshold) == y).mean()),
                      "threshold": threshold, "n": int(len(y))}
        print("OOF:", {k: round(v, 4) if isinstance(v, float) else v for k, v in oof_report.items()})

    torch.save({"format": "dfbundle-v1", "members": members, "threshold": threshold,
                "tta_hflip": True, "oof": oof_report}, args.out)
    print(f"wrote {args.out} ({len(members)} members, threshold={threshold:.4f})")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile code/inference_script.py
"""
Deepfake ML Hackathon — inference script.

    python inference_script.py --test_dir ./test/images/ --weights ./model_weights.pt --output predictions.csv

Self-contained: needs only torch, torchvision, timm, pandas, Pillow (no repo imports, no downloads).
`--weights` is a bundle produced by build_bundle.py: one or more fine-tuned backbones whose LoRA
adapters were merged into the weights, plus the OOF-selected decision threshold.
"""
import argparse
import os
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

IMG_EXTS = {".jpg", ".jpeg", ".png"}
LOAD_SIZE = 224  # test images are 224x224; anything else is resized once on load


class Detector(nn.Module):
    """Same architecture as training (src/model.py), minus LoRA (already merged into weights)."""

    def __init__(self, arch, img_size, feat_layers=None):
        super().__init__()
        self.backbone = timm.create_model(arch, pretrained=False, num_classes=0, img_size=img_size)
        self.feat_layers = list(feat_layers) if feat_layers else None
        dim = self.backbone.num_features
        if self.feat_layers:  # multi-layer CLS features
            self.layer_norms = nn.ModuleList(nn.LayerNorm(dim) for _ in self.feat_layers)
            dim *= len(self.feat_layers)
        self.head = nn.Sequential(nn.Dropout(0.0), nn.Linear(dim, 1))

    def features(self, x):
        if not self.feat_layers:
            return self.backbone(x)
        b = self.backbone
        x = b.norm_pre(b.patch_drop(b._pos_embed(b.patch_embed(x))))
        wanted, cls = set(self.feat_layers), []
        for i, blk in enumerate(b.blocks):
            x = blk(x)
            if i in wanted:
                cls.append(x[:, 0])
        return torch.cat([ln(c) for ln, c in zip(self.layer_norms, cls)], dim=1)

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)


class TestDataset(Dataset):
    def __init__(self, test_dir):
        self.paths = sorted(p for p in Path(test_dir).iterdir() if p.suffix.lower() in IMG_EXTS)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if img.size != (LOAD_SIZE, LOAD_SIZE):
            img = img.resize((LOAD_SIZE, LOAD_SIZE), Image.BICUBIC)
        x = torch.from_numpy(np.asarray(img, dtype=np.uint8).copy()).permute(2, 0, 1)
        return x, self.paths[i].stem


def load_members(bundle, device, dtype):
    members = []
    for m in bundle["members"]:
        net = Detector(m["arch"], m["img_size"], m.get("feat_layers"))
        net.load_state_dict({k: v.float() for k, v in m["state_dict"].items()})
        net = net.to(device=device, dtype=dtype).eval()
        mean = torch.tensor(m["mean"], device=device, dtype=dtype).view(1, 3, 1, 1)
        std = torch.tensor(m["std"], device=device, dtype=dtype).view(1, 3, 1, 1)
        members.append((net, m["img_size"], mean, std, float(m.get("weight", 1.0))))
    return members


@torch.inference_mode()
def predict(members, loader, device, dtype, tta_hflip):
    ids, probs = [], []
    wsum = sum(w for *_, w in members)
    for x, batch_ids in loader:
        x = x.to(device, non_blocking=True).to(dtype).div_(255.0)
        logit = torch.zeros(x.shape[0], device=device, dtype=torch.float32)
        for net, size, mean, std, w in members:
            xi = x if size == x.shape[-1] else F.interpolate(
                x, size=(size, size), mode="bicubic", align_corners=False, antialias=True)
            xi = (xi - mean) / std
            out = net(xi).float()
            if tta_hflip:
                out = 0.5 * (out + net(torch.flip(xi, dims=[3])).float())
            logit += w * out
        probs.append(torch.sigmoid(logit / wsum).cpu().numpy())
        ids.extend(batch_ids)
    return ids, np.concatenate(probs) if probs else np.zeros(0)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--test_dir", required=True, help="Path to test images folder")
    parser.add_argument("--weights", required=True, help="Path to model weight file")
    parser.add_argument("--output", default="predictions.csv", help="Output CSV path")
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--num_workers", type=int, default=min(8, os.cpu_count() or 1))
    args = parser.parse_args()

    torch.backends.cudnn.benchmark = False
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    bundle = torch.load(args.weights, map_location="cpu", weights_only=False)
    members = load_members(bundle, device, dtype)
    threshold = float(bundle.get("threshold", 0.5))
    print(f"device={device} members={[m['arch'] for m in bundle['members']]} threshold={threshold:.4f}")
    del bundle

    ds = TestDataset(args.test_dir)
    loader = DataLoader(ds, batch_size=args.batch_size, shuffle=False,
                        num_workers=args.num_workers, pin_memory=device == "cuda")
    ids, conf = predict(members, loader, device, dtype, tta_hflip=True)

    conf = np.clip(conf, 0.0, 1.0)
    out = pd.DataFrame({"image_id": ids, "label": (conf >= threshold).astype(int),
                        "confidence": np.round(conf, 6)})
    assert len(out) == len(ds) and out.image_id.is_unique
    out.to_csv(args.output, index=False)
    print(f"wrote {len(out)} predictions -> {args.output} (predicted fake: {out.label.mean():.3f})")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile code/requirements.txt
# Python 3.10. Tested versions; timm is pinned exactly because state-dict keys depend on it.
torch>=2.1,<3
torchvision>=0.16
timm==1.0.31
numpy>=2.0,<3
pandas>=2.0
scikit-learn>=1.3
Pillow>=10.0

## Locate the dataset

In [ ]:
import os, glob
from pathlib import Path

def find_dir(pred):
    for root, dirs, files in os.walk("/kaggle/input"):
        if pred(Path(root), dirs, files):
            return Path(root)
    raise FileNotFoundError("dataset layout not found under /kaggle/input")

train_dir = find_dir(lambda r, d, f: "real" in d and "fake" in d)
test_dir  = find_dir(lambda r, d, f: r.name == "images" and len(f) > 100)
folds_csv = next(Path("/kaggle/input").rglob("folds.csv"))

data = Path("/kaggle/working/data"); data.mkdir(exist_ok=True)
if not (data / "train").exists():
    (data / "train").symlink_to(train_dir)
print("train:", train_dir, "| test:", test_dir, "| folds:", folds_csv)
print({k: len(os.listdir(train_dir / k)) for k in ("real", "fake")}, "test:", len(os.listdir(test_dir)))

## Train

In [ ]:
%cd /kaggle/working/code
for run, extra in RUNS.items():
    for fold in FOLDS:
        print(f"\n######## {run} | fold {fold} ########", flush=True)
        !python train.py --data_dir /kaggle/working/data --folds_csv {folds_csv} --fold {fold} --arch {ARCH} --workers 4 --run_name {run} --out_dir /kaggle/working/outputs {COMMON} {extra}

In [ ]:
import json, os, pandas as pd
rows = []
for run in RUNS:
    for fold in FOLDS:
        d = f"/kaggle/working/outputs/{run}/fold{fold}"
        if not os.path.exists(f"{d}/robustness.json"):
            print(f"!! {run} fold {fold}: no results (run failed?)"); continue
        print(f"== {run} | fold {fold}"); print(pd.read_csv(f"{d}/history.csv").round(4).to_string(index=False))
        r = json.load(open(f"{d}/robustness.json"))
        rows.append({"run": run, "fold": fold, "clean": r["clean"], "mean_degraded": r["mean_degraded"],
                     "worst_degraded": r["worst_degraded"]})
summary = pd.DataFrame(rows)
summary.to_csv("/kaggle/working/summary.csv", index=False)
print("\n=== SUMMARY (baseline fold-0 clean AUC 0.9134) ===")
print(summary.groupby("run")[["clean", "mean_degraded", "worst_degraded"]].mean().round(4).sort_values("clean", ascending=False))

## Bundle + timed test inference (T4, same script the judges run)

In [ ]:
import time
if RUN_TEST_INFERENCE and len(summary):
    best_run = summary.groupby("run")["clean"].mean().idxmax()
    print("bundling best run:", best_run)
    out = f"/kaggle/working/outputs/{best_run}"
    !python build_bundle.py --members "{out}/fold*/member.pt" --oof "{out}/fold*/oof.csv" --out /kaggle/working/model_weights.pt
    t0 = time.time()
    !python inference_script.py --test_dir {test_dir} --weights /kaggle/working/model_weights.pt --output /kaggle/working/predictions.csv
    print(f"inference wall time: {time.time() - t0:.1f}s (target < 300s)")
    !head -5 /kaggle/working/predictions.csv; wc -l /kaggle/working/predictions.csv